# Experimenting with LLM's to provide uncertainty explanations between quantiles

**The Goal**: Extend the horizon architecture to predict multiple quantiles and use SHAP with Groq LLM to explain the model's uncertainty with specific predictions


## Setup and config

In [33]:
# --- 1. CONFIGURATION & IMPORTS ---
import polars as pl
import lightgbm as lgb
import numpy as np
import pandas as pd
import shap
import os
from groq import Groq
import warnings
warnings.filterwarnings('ignore')

# --- 2. GLOBAL SETTINGS ---
DATA_PATH = "../../backend/data/processed/m5_improved.parquet" # Walmart M5 Dataset
NUM_FOLDS = 1 # Reduced for fast experimentation
QUANTILES = [0.10, 0.50, 0.90] # Lower bound, Median, Upper bound (Safety Stock)
RANDOM_SEED = 42

# Initialize Groq Client (Ensure GROQ_API_KEY is in your environment variables)
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Define features once
FEATURES = [
    'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 
    'wday', 'month', 'sell_price', 'price_norm',
    'price_momentum_7d', 'price_momentum_28d',
    'lag_28', 'roll_mean_7_lag_28', 'roll_mean_28_lag_28',
    'masked_roll_mean_28_lag_28', 'ema_lag_28', 'days_since_last_sale_lag_28',
    'horizon_day', 
    'target_wday' # <--- THE NEW MAGIC FEATURE
]

CAT_FEATURES = ["item_id", "dept_id", "cat_id", "store_id", "state_id", "target_wday"]

HORIZONS = range(1, 29)

def pinball_loss(y_true, y_pred, alpha):
    """Calculates the Quantile (Pinball) Loss."""
    error = y_true - y_pred
    return np.mean(np.maximum(alpha * error, (alpha - 1) * error))

## Data injestion and Feature Engineering

In [35]:
## Data Ingestion & Time-Series Feature Engineering
print("📂 Loading Data and Engineering Features via Polars...")

df_full = pl.read_parquet(DATA_PATH)
max_d = df_full["d"].max()
train_start_d = max_d - (28 * NUM_FOLDS) - 365 - 56 

# Downsample for fast prototyping
np.random.seed(RANDOM_SEED)
sampled_items = np.random.choice(df_full.select("item_id").unique().to_series().to_list(), size=int(df_full.select("item_id").n_unique() * 0.05), replace=False)
df_base = df_full.filter((pl.col("d") >= train_start_d) & pl.col("item_id").is_in(sampled_items)).sort(["store_id", "item_id", "d"])

df_base = df_base.with_columns([
    (pl.col("sell_price") / pl.col("sell_price").shift(7).over(["store_id", "item_id"])).alias("price_momentum_7d"),
    (pl.col("sell_price") / pl.col("sell_price").shift(28).over(["store_id", "item_id"])).alias("price_momentum_28d"),
    pl.col("sales").shift(28).over(["store_id", "item_id"]).alias("lag_28")
]).with_columns(
    pl.int_range(1, pl.len() + 1).over(["store_id", "item_id"]).alias("row_nr")
)

df_base = df_base.with_columns([
    pl.col("lag_28").rolling_mean(window_size=7).over(["store_id", "item_id"]).alias("roll_mean_7_lag_28"),
    pl.col("lag_28").rolling_mean(window_size=28).over(["store_id", "item_id"]).alias("roll_mean_28_lag_28"),
    pl.when(pl.col("lag_28") > 0).then(pl.col("lag_28")).otherwise(None)
      .rolling_mean(window_size=28, min_periods=1).forward_fill()
      .over(["store_id", "item_id"]).alias("masked_roll_mean_28_lag_28"),
    pl.col("lag_28").ewm_mean(alpha=0.1, ignore_nulls=True).over(["store_id", "item_id"]).alias("ema_lag_28"),
    pl.when(pl.col("lag_28") > 0).then(pl.col("row_nr")).otherwise(None)
      .forward_fill().over(["store_id", "item_id"]).alias("last_sale_row")
])

df_base = df_base.with_columns([
    (pl.col("row_nr") - pl.col("last_sale_row")).fill_null(0).alias("days_since_last_sale_lag_28")
]).drop(["row_nr", "last_sale_row"]).drop_nulls(subset=["roll_mean_28_lag_28", "price_momentum_28d"])

# Only cast string categorical columns that currently exist in df_base
for col in CAT_FEATURES:
    if col in df_base.columns:
        df_base = df_base.with_columns(pl.col(col).cast(pl.String).cast(pl.Categorical).to_physical().alias(col))  
    
# --- 2. DATA ENGINEERING UPDATE ---
# (Keep your existing df_base feature engineering code the same up until creating df_train)

print("🔄 Stacking 28 Horizons for Global Training...")
horizon_dfs = [
    df_base.with_columns([
        pl.col("sales").shift(-h).over(["store_id", "item_id"]).alias("target"), 
        pl.lit(h).cast(pl.Int16).alias("horizon_day"),
        
        # Calculates the future day of the week (1-7) without breaking the cycle
        ((pl.col("wday") + h - 1) % 7 + 1).cast(pl.Int8).alias("target_wday") 
        
    ]).drop_nulls(subset=["target"]) 
    for h in HORIZONS
]

df_train = pl.concat(horizon_dfs)
X_train = df_train.select(FEATURES).to_pandas()
y_train = df_train.select("target").to_series().to_numpy()

📂 Loading Data and Engineering Features via Polars...
🔄 Stacking 28 Horizons for Global Training...


## Multi-quantile training

In [36]:
## Multi-Quantile LightGBM Training
print("🚀 Training LightGBM Models for Quantiles:", QUANTILES)

models = {}
train_data = lgb.Dataset(X_train, y_train, categorical_feature=CAT_FEATURES, free_raw_data=False)

for q in QUANTILES:
    print(f"Training Q{int(q*100)} model...")
    models[q] = lgb.train(
        {'objective': 'quantile', 'alpha': q, 'learning_rate': 0.1, 'num_leaves': 64, 'verbosity': -1}, 
        train_data, 
        num_boost_round=100
    )
    
print("✅ All quantile models trained successfully.")

🚀 Training LightGBM Models for Quantiles: [0.1, 0.5, 0.9]
Training Q10 model...
Training Q50 model...
Training Q90 model...
✅ All quantile models trained successfully.


## SHAP uncertainty and Groq LLM Explanation

In [42]:
# ==========================================
# CELL 5: 28-DAY MULTI-QUANTILE DEBUGGING DASHBOARD
# ==========================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import numpy as np
import shap

print("🎛️ Loading 28-Day Multi-Quantile Debugging Dashboard...")

# 1. Pre-Initialize SHAP Explainers
explainer_q10 = shap.TreeExplainer(models[0.10])
explainer_q50 = shap.TreeExplainer(models[0.50])
explainer_q90 = shap.TreeExplainer(models[0.90])

# 2. Extract unique items and stores
unique_items = sorted(df_train['item_id'].drop_nulls().unique().to_list())
unique_stores = sorted(df_train['store_id'].drop_nulls().unique().to_list())

# 3. Core Selection Controls
item_dropdown = widgets.Dropdown(options=unique_items, description='Item ID:')
store_dropdown = widgets.Dropdown(options=unique_stores, description='Store ID:')
analyze_button = widgets.Button(description='Debug 28-Day Forecast', button_style='warning', layout=widgets.Layout(width='200px'))
output = widgets.Output()

def on_analyze_clicked(b):
    with output:
        clear_output(wait=True)
        TARGET_ITEM = item_dropdown.value
        TARGET_STORE = store_dropdown.value
        
        # Pull from df_base to establish our "Current Day" (d)
        matching_rows_pl = df_base.filter(
            (pl.col("item_id") == TARGET_ITEM) & (pl.col("store_id") == TARGET_STORE)
        )
        
        if matching_rows_pl.height == 0:
            print("⚠️ No data found for this specific Item/Store combination. Please try another.")
            return
            
        # Get the row representing 'Day 0' (the last day of available history before the forecast)
        # We need to drop the last 28 days of the dataset to have actuals to compare against
        max_available_d = matching_rows_pl["d"].max()
        current_d = max_available_d - 28
        
        base_row = matching_rows_pl.filter(pl.col("d") == current_d).to_pandas().iloc[[-1]]
        
        print(f"⚙️ Debugging 28-Day Forecast for Item {TARGET_ITEM} at Store {TARGET_STORE}...")
        print(f"📅 Historical Cutoff: Day {current_d} | Predicting: Days {current_d + 1} to {current_d + 28}")
        
        # --- BUILD 28-DAY INFERENCE DATAFRAME ---
        # 1. Safely duplicate the row while PRESERVING categorical dtypes
        future_X = pd.concat([base_row.iloc[[0]]] * 28, ignore_index=True)
        
        # 2. Inject the horizon_day (1 to 28)
        future_X['horizon_day'] = range(1, 29)
        
        # 3. Inject target_wday natively as integers (matching training data)
        start_wday = int(base_row['wday'].iloc[0])
        future_X['target_wday'] = [(start_wday + i - 1) % 7 + 1 for i in range(1, 29)]
        
        # 4. Align features perfectly with the model
        future_X = future_X[FEATURES] 
        future_days = np.arange(current_d + 1, current_d + 29)
        
        # --- PREDICTIONS ---
        pred_q10 = models[0.10].predict(future_X)
        pred_q50 = models[0.50].predict(future_X)
        pred_q90 = models[0.90].predict(future_X)
        
 # --- FETCH ACTUALS ---
        # Fetch exactly the last 28 days of history for a symmetric 56-day comparison
        history = matching_rows_pl.filter(pl.col("d") <= current_d).select(["d", "sales"]).tail(28).to_pandas()
        
        actual_future = matching_rows_pl.filter(
            (pl.col("d") > current_d) & (pl.col("d") <= current_d + 28)
        ).select(["d", "sales"]).to_pandas()

        # --- VISUALIZATION ---
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [2, 1.2]})
        
        # -----------------------------------------
        # PLOT 1 (Left): Daily Forecast vs Actuals
        # -----------------------------------------
        if not history.empty:
            ax1.plot(history['d'], history['sales'], color='black', alpha=0.4, label="28-Day History")
            
        ax1.fill_between(future_days, pred_q10, pred_q90, color='red', alpha=0.15, label="Uncertainty Cone (Q10-Q90)")
        ax1.plot(future_days, pred_q50, marker='o', linestyle='--', color='#1f77b4', linewidth=2, label="Q50 Forecast")
                     
        if not actual_future.empty:
            ax1.plot(actual_future['d'], actual_future['sales'], marker='x', color='#2ca02c', markersize=6, linestyle='-', linewidth=2, label="Actual Sales")

        ax1.set_title(f"Daily Forecast vs Actuals\nItem: {TARGET_ITEM} | Store: {TARGET_STORE}", fontweight='bold')
        ax1.set_xlabel("Day (d)")
        ax1.set_ylabel("Units Sold")
        ax1.axvline(x=current_d, color='gray', linestyle=':', label='Forecast Start')
        ax1.legend(loc='upper left')
        ax1.grid(True, alpha=0.3)
        
        # -----------------------------------------
        # PLOT 2 (Right): Continuous Cumulative Trend
        # -----------------------------------------
        # 1. Calculate History Cumulative
        if not history.empty:
            cum_hist = np.cumsum(history['sales'])
            ax2.plot(history['d'], cum_hist, color='black', linewidth=2, label="Historical Cumulative")
            hist_end_val = cum_hist.iloc[-1]
        else:
            hist_end_val = 0

        # 2. Calculate Forecast Cumulative (Anchored to the end of history)
        cum_q10 = np.cumsum(pred_q10) + hist_end_val
        cum_q50 = np.cumsum(pred_q50) + hist_end_val
        cum_q90 = np.cumsum(pred_q90) + hist_end_val
        
        ax2.fill_between(future_days, cum_q10, cum_q90, color='orange', alpha=0.2, label="Cumulative Spread")
        ax2.plot(future_days, cum_q50, color='#ff7f0e', linewidth=2.5, label="Cumulative Q50")
        
        # 3. Calculate Actual Future Cumulative (Anchored to the end of history)
        if not actual_future.empty:
            cum_actual = np.cumsum(actual_future['sales']) + hist_end_val
            ax2.plot(actual_future['d'], cum_actual, marker='x', color='#2ca02c', markersize=6, linestyle='--', linewidth=2, label="Cumulative Actuals")

        ax2.set_title("56-Day Cumulative Trend", fontweight='bold')
        ax2.set_xlabel("Day (d)")
        ax2.set_ylabel("Total Units")
        ax2.axvline(x=current_d, color='gray', linestyle=':', label='Forecast Start')
        ax2.legend(loc='upper left')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # --- SHAP UNCERTAINTY ATTRIBUTION (Peak Risk Day) ---
        spreads = pred_q90 - pred_q10
        peak_risk_idx = np.argmax(spreads)
        peak_risk_day = future_days[peak_risk_idx]
        peak_horizon = peak_risk_idx + 1
        
        print(f"\n🔍 Peak Uncertainty detected on Horizon Day {peak_horizon} (Day {peak_risk_day}). Analyzing drivers...")
        
        peak_X_sample = future_X.iloc[[peak_risk_idx]]
        
        shap_q10 = explainer_q10(peak_X_sample)
        shap_q50 = explainer_q50(peak_X_sample)
        shap_q90 = explainer_q90(peak_X_sample)
        
        uncertainty_drivers = shap_q90.values[0] - shap_q10.values[0]
        impact_df = pd.DataFrame({
            'Feature': future_X.columns,
            'Value': peak_X_sample.iloc[0].values,
            'Uncertainty_Contribution': uncertainty_drivers
        }).sort_values(by='Uncertainty_Contribution', ascending=False)

        top_3 = impact_df.head(3)
        
        print("\n📈 Top 3 Drivers of Peak Uncertainty:")
        for _, row in top_3.iterrows():
            val_display = f"{row['Value']:.4f}" if isinstance(row['Value'], float) else f"{row['Value']}"
            print(f"- {row['Feature']}: {row['Uncertainty_Contribution']:.2f} unit spread (Value: {val_display})")
        
        # --- GROQ LLM CALL ---
        prompt = f"""
        You are an expert supply chain data scientist analyzing a risk quantification model predicting 28 days of retail sales.
        
        The model showed the highest 'demand uncertainty' (the gap between the 10th and 90th percentile forecast) on exactly horizon day {peak_horizon}.

        For this specific item on this specific day, the top 3 features driving the uncertainty spread are:
        1. '{top_3.iloc[0]['Feature']}' (Value: {top_3.iloc[0]['Value']})
        2. '{top_3.iloc[1]['Feature']}' (Value: {top_3.iloc[1]['Value']})
        3. '{top_3.iloc[2]['Feature']}' (Value: {top_3.iloc[2]['Value']})

        In two to three short, professional sentences, explain to a retail manager why these specific features might cause the model's prediction to destabilize specifically at day {peak_horizon} of the forecast. Do not use complex math jargon.
        """
        
        try:
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama-3.1-8b-instant",
            )
            print("\n🤖 Groq Insight (Drivers of Peak Uncertainty):")
            print(response.choices[0].message.content)
        except Exception as e:
            print(f"\n⚠️ Groq LLM Error: Ensure your API key is active. ({e})")
            
        print("\n" + "="*50)
        print(f"📊 SHAP WATERFALL PLOTS FOR HORIZON DAY {peak_horizon}")
        print("="*50 + "\n")
        
        print("🔻 Q10 Model (Pessimistic Scenario Drivers)")
        plt.figure(figsize=(8, 4))
        shap.plots.waterfall(shap_q10[0], max_display=8, show=False)
        plt.tight_layout()
        plt.show()

        print("🔺 Q90 Model (Spike/Safety Stock Drivers)")
        plt.figure(figsize=(8, 4))
        shap.plots.waterfall(shap_q90[0], max_display=8, show=False)
        plt.tight_layout()
        plt.show()

analyze_button.on_click(on_analyze_clicked)

display(widgets.HBox([item_dropdown, store_dropdown, analyze_button]))
display(output)

🎛️ Loading 28-Day Multi-Quantile Debugging Dashboard...


Output()